# Trying the ByT5 Model for Early Modern Print Abbreviation Expansion

This notebook demonstrates how to load and use the [mschonhardt/sal_abbreviations](https://huggingface.co/mschonhardt/sal_abbreviations) model to expand abbreviations in early modern prints.

In [ ]:
!pip install -q transformers torch datasets pandas

In [1]:
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer
import pandas as pd
import json
import random
import os

model_name = "mschonhardt/sal_abbreviations"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model and tokenizer: {model_name} on {device}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)
print("Model loaded successfully.")

Loading model and tokenizer: mschonhardt/sal_abbreviations on cuda...


tokenizer_config.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/815 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

Model loaded successfully.


In [4]:
def expand_abbreviation(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example usage
test_text = "lis est, ipse intrabit in regnum c\u00e6lor\u0169. Matt. 7."
print(f"Source: {test_text}")
print(f"Expanded: {expand_abbreviation(test_text)}")

Source: lis est, ipse intrabit in regnum cælorũ. Matt. 7.
Expanded: lis est, ipse intrabit in regnum cælorum. Matt. 7.


We can also integrate this into a tei pipeline (here in a very simplified manner).

In [12]:
import difflib

# Assuming 'tokenizer', 'model', and 'device' are already initialized in your notebook

def expand_abbreviation(text):
    """Generates the expanded text using the Seq2Seq model."""
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def generate_tei_markup(source_text, expanded_text):
    """
    Aligns source and expanded text at the word level and generates TEI XML markup 
    for abbreviations and expansions.
    """
    # Tokenizing by whitespace to ensure semantic word-level alignment
    source_words = source_text.split()
    expanded_words = expanded_text.split()
    
    matcher = difflib.SequenceMatcher(None, source_words, expanded_words)
    tei_result = []
    
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal':
            # Text is identical; append as is
            tei_result.append(" ".join(source_words[i1:i2]))
            
        elif tag == 'replace':
            # Text was altered; wrap in TEI choice element
            abbr = " ".join(source_words[i1:i2])
            expan = " ".join(expanded_words[j1:j2])
            tei_result.append(f'<choice><abbr>{abbr}</abbr><expan resp="model">{expan}</expan></choice>')
            
        elif tag == 'delete':
            # The model omitted text; track the deletion
            deleted_text = " ".join(source_words[i1:i2])
            tei_result.append(f'<del resp="model">{deleted_text}</del>')
            
        elif tag == 'insert':
            # The model hallucinated or added text; track the insertion
            added_text = " ".join(expanded_words[j1:j2])
            tei_result.append(f'<add resp="model">{added_text}</add>')

    return " ".join(tei_result)

# Example usage
test_text = "lis est, ipse intrabit in regnum c\u00e6lor\u0169. Matt. 7."
expanded_text = expand_abbreviation(test_text)
tei_output = generate_tei_markup(test_text, expanded_text)

print("--- Results ---")
print(f"Source String:   {test_text}")
print(f"Model Output:    {expanded_text}")
print(f"TEI XML Markup:  {tei_output}")

--- Results ---
Source String:   lis est, ipse intrabit in regnum cælorũ. Matt. 7.
Model Output:    lis est, ipse intrabit in regnum cælorum. Matt. 7.
TEI XML Markup:  lis est, ipse intrabit in regnum <choice><abbr>cælorũ.</abbr><expan resp="model">cælorum.</expan></choice> Matt. 7.


## Loading the Dataset

We can load the dataset either from the local `test.jsonl` file or directly from Hugging Face.

In [5]:
from datasets import load_dataset

data_path = "test.jsonl"

if os.path.exists(data_path):
    print(f"Loading local dataset from {data_path}...")
    data = []
    with open(data_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    df = pd.DataFrame(data)
else:
    print("Local file not found. Loading from Hugging Face...")
    dataset = load_dataset("mschonhardt/sal_abbreviations", split="test")
    df = dataset.to_pandas()

print(f"Loaded {len(df)} lines.")
df.head()

Loading local dataset from test.jsonl...
Loaded 357514 lines.


,id,doc_id,file_name,parent_id,parent_type,line_index,source,target,end_lb_break_no,fixed_from_lb,lb_id,facs_id,image_url,has_abbreviation,target_original
0,W0008-07-0003-he-03e8:l0001,W0008_Vol07,W0008_Vol07.xml,W0008-07-0003-he-03e8,head,1,INDEX,INDEX,False,False,W0008-07-0003-lb-0001,facs:W0008-G-0003,https://facs.salamanca.school/iiif/image/W0008...,False,NaN
1,W0008-07-0003-he-03e8:l0002,W0008_Vol07,W0008_Vol07.xml,W0008-07-0003-he-03e8,head,2,"DISPVTATIONVM,","DISPVTATIONVM,",False,False,W0008-07-0003-lb-0002,facs:W0008-G-0003,https://facs.salamanca.school/iiif/image/W0008...,False,NaN
2,W0008-07-0003-he-03e8:l0003,W0008_Vol07,W0008_Vol07.xml,W0008-07-0003-he-03e8,head,3,QVAE HOC TOMO,QVAE HOC TOMO,False,False,W0008-07-0003-lb-0003,facs:W0008-G-0003,https://facs.salamanca.school/iiif/image/W0008...,False,NaN
3,W0008-07-0003-he-03e8:l0004,W0008_Vol07,W0008_Vol07.xml,W0008-07-0003-he-03e8,head,4,SEXTO CON,SEXTO CON,True,False,W0008-07-0003-lb-0004,facs:W0008-G-0003,https://facs.salamanca.school/iiif/image/W0008...,False,NaN
4,W0008-07-0003-he-03e8:l0005,W0008_Vol07,W0008_Vol07.xml,W0008-07-0003-he-03e8,head,5,TINENTVR.,TINENTVR.,None,False,W0008-07-0003-lb-0005,facs:W0008-G-0003,https://facs.salamanca.school/iiif/image/W0008...,False,NaN


## Sampling and Expansion

We will now select 10 lines from three categories:
1. Randomly selected lines.
2. Lines flagged as containing abbreviations (`has_abbreviation == True`).
3. Lines flagged as NOT containing abbreviations (`has_abbreviation == False`).

In [6]:
def run_demonstration(subset_df, title):
    print(f"\n--- {title} ---")
    results = []
    for _, row in subset_df.iterrows():
        source = row["source"]
        target = row["target"]
        prediction = expand_abbreviation(source)
        results.append({
            "Source": source,
            "Ground Truth": target,
            "Prediction": prediction
        })
    return pd.DataFrame(results)

# 1. 10 Randomly selected lines
df_random = df.sample(10)
res_random = run_demonstration(df_random, "10 Random Lines")

# 2. 10 flagged as abbreviation
df_abbr = df[df["has_abbreviation"] == True].sample(min(10, len(df[df["has_abbreviation"] == True])))
res_abbr = run_demonstration(df_abbr, "10 Abbreviated Lines")

# 3. 10 NOT flagged as abbreviation
df_no_abbr = df[df["has_abbreviation"] == False].sample(min(10, len(df[df["has_abbreviation"] == False])))
res_no_abbr = run_demonstration(df_no_abbr, "10 Non-Abbreviated Lines")


--- 10 Random Lines ---

--- 10 Abbreviated Lines ---

--- 10 Non-Abbreviated Lines ---


### Results: 10 Random Lines

In [7]:
res_random

,Source,Ground Truth,Prediction
0,dis accidere potest vt quis sit acceptionis cau,dis accidere potest vt quis sit acceptionis cau,dis accidere potest vt quis sit acceptionis cau
1,cuniæ; ergo cum ista commoditas sit pretio,cuniæ; ergo cum ista commoditas sit pretio,cuniæ; ergo cum ista commoditas sit pretio
2,"in univerſo Orbe in teſtimonium gẽtibus,","in univerſo Orbe in teſtimonium gentibus,","in univerſo Orbe in teſtimonium gentibus,"
3,"tegritatem sacrificij referenda est, quo pacto...","tegritatem sacrificij referenda est, quo pacto...","tegritatem sacrificij referenda est, quo pacto..."
4,"etiam rectæ, non oriri impedimentum, ex quo","etiam rectæ, non oriri impedimentum, ex quo","etiam rectæ, non oriri impedimentum, ex quo"
5,"gava quanto podia, que tuvieſſe eſpecial","gava quanto podia, que tuvieſſe eſpecial","gava quanto podia, que tuvieſſe eſpecial"
6,"quæ peregrè, & potissimè quæ maritimis pe","quæ peregrè, & potissimè quæ maritimis pe","quæ peregrè, & potissimè quæ maritimis pe"
7,"ſtigia reperi, videntur. Ft numer.","ſtigia reperi, videntur. Ft numer.","ſtigia reperi, videntur. Ft numer."
8,bata contra veritatem quam cognoscit pri,bata contra veritatem quam cognoscit pri,bata contra veritatem quam cognoscit pri
9,"¶ Atque inde liquet tertij responsio, quo","¶ Atque inde liquet tertij responsio, quo","¶ Atque inde liquet tertij responsio, quo"


### Results: 10 Abbreviated Lines

In [8]:
res_abbr

,Source,Ground Truth,Prediction
0,"obligant, sub moralibus præceptis ea etiam cō","obligant, sub moralibus præceptis ea etiam com","obligant, sub moralibus præceptis ea etiam con"
1,illud Apost. 1. ad Tim. 5. Qui bene p̃sunt pre,illud Apost. 1. ad Tim. 5. Qui bene presunt pre,illud Apost. 1. ad Tim. 5. Qui bene presunt pre
2,"ueri, ad perceptorum fructuum restitutionẽ","ueri, ad perceptorum fructuum restitutionem","ueri, ad perceptorum fructuum restitutionem"
3,"Respondetur, quòd nimis durũ esset obliga","Respondetur, quòd nimis durum esset obliga","Respondetur, quòd nimis durum esset obliga"
4,tanquàm mouente & in altero tāquàm mo,tanquàm mouente & in altero tanquam mo,tanquàm mouente & in altero tanquàm mo
5,quando coniecturæ non nimiũ vrgent. Vn,quando coniecturæ non nimium vrgent. Vn,quando coniecturæ non nimium vrgent. Vn
6,tur. Et hoc verificatur tam in genere substā,tur. Et hoc verificatur tam in genere substan,tur. Et hoc verificatur tam in genere substan
7,ditori. De amico etiā insigni & speciali pro,ditori. De amico etiam insigni & speciali pro,ditori. De amico etiam insigni & speciali pro
8,fit credibile: tantoq́; minùs quanto illas gen,fit credibile: tantoque minùs quanto illas gen,fit credibile: tantoque minùs quanto illas gen
9,nat: atq; adeò in hoc rationis actu consistit,nat: atque adeò in hoc rationis actu consistit,nat: atque adeò in hoc rationis actu consistit


### Results: 10 Non-Abbreviated Lines

In [9]:
res_no_abbr

,Source,Ground Truth,Prediction
0,"suspicio, & in verbo iudicium temerarium,","suspicio, & in verbo iudicium temerarium,","suspicio, & in verbo iudicium temerarium,"
1,Horę quę in communi soluuntur in ecclesia,Horę quę in communi soluuntur in ecclesia,Horę quę in communi soluuntur in ecclesia
2,"tificibus Prædecessoribus nostris, etiam per s...","tificibus Prædecessoribus nostris, etiam per s...","tificibus Prædecessoribus nostris, etiam per s..."
3,"Veris & legitimis, col. 7. verſ. Sed hìc","Veris & legitimis, col. 7. verſ. Sed hìc","Veris & legitimis, col. 7. verſ. Sed hìc"
4,"D. de doli except. l. militis codicillis,","D. de doli except. l. militis codicillis,","D. de doli except. l. militis codicillis,"
5,expeditionibus Hiſpanorum ad Novum,expeditionibus Hiſpanorum ad Novum,expeditionibus Hiſpanorum ad Novum
6,ri vltra valorem quo ceu numisma æstima,ri vltra valorem quo ceu numisma æstima,ri vltra valorem quo ceu numisma æstima
7,2502. e,2502. e,2502. e
8,Vtrùm licitum sit monetarum cam,Vtrùm licitum sit monetarum cam,Vtrùm licitum sit monetarum cam
9,"& Angelorum, & reliquarum creatu","& Angelorum, & reliquarum creatu","& Angelorum, & reliquarum creatu"
